# Spine XR Augmentation — Project 3 Colab Runner

Google Colab Pro+ (A100) üzerinde koşulmak üzere tasarlandı. Tüm ağır eğitim buradan yürütülür.

## Sıralama

1. Bootstrap (Drive mount, repo + dataset.rar'ı SSD'ye çek, outputs'u Drive'a sembolik bağla, pip install)
2. `01_audit` → `02_data_splitter`
3. `03_train_classifier` — 4 Case × 2 Backbone (VGG16, InceptionV3) — baseline (no aug)
4. (Sonraki milestone'larda) Phase 04 traditional, Phase 05–07 WGAN, Phase 08 hybrid, Phase 09 final report

## 1. Bootstrap

In [ ]:
# 1. Drive Mount
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# 2. Çalışma Alanını Yerel SSD'de Ayarla (A100'ün maksimum hızı için)
LOCAL_ROOT = Path('/content/spine-xr-augmentation-study')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(LOCAL_ROOT)

# 3. Kodları Drive'dan Yerele Kopyala
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/spine-xr-augmentation-study')
!cp -r {DRIVE_REPO_PATH}/* .

# 4. Dataset'i SSD'ye Çek ve Aç (Dataset Drive'da .rar olarak durmalı)
# Klasör Adı "dataset" olmalı — configs/base.yaml relatif `dataset/...` yolları kullanır.
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/dataset.rar')
!unrar x -o+ {DRIVE_DATASET_PATH} {LOCAL_ROOT}/

# 5. Çıktıların (Outputs) Kaybolmaması İçin Drive'a Bağla
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/spine-xr-augmentation-study/outputs')
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

if os.path.exists('outputs') and not os.path.islink('outputs'):
    shutil.rmtree('outputs')
elif os.path.islink('outputs'):
    os.remove('outputs')
os.symlink(DRIVE_OUTPUTS, 'outputs')

print(f"Çalışma dizini (SSD): {os.getcwd()}")
!ls -l

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 2. Audit + Splits

In [ ]:
!python scripts/01_audit.py --config configs/base.yaml
!python scripts/02_data_splitter.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/01_audit/audit_report.md
!echo '---'
!cat outputs/02_splits/splits_summary.md

## 3. Baseline classifier — 4 Case × 2 Backbone

### Smoke Test

In [ ]:
# SMOKE: en küçük case (case_4) + VGG16 + 1 epoch — pipeline çalışıyor mu kontrolü
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter vgg16 \
    --epochs 1 \
    --out-tag 03_smoke

### Case 1 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_1 \
    --backbones-filter vgg16

### Case 1 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_1 \
    --backbones-filter inception_v3

### Case 2 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_2 \
    --backbones-filter vgg16

### Case 2 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_2 \
    --backbones-filter inception_v3

### Case 3 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_3 \
    --backbones-filter vgg16

### Case 3 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_3 \
    --backbones-filter inception_v3

### Case 4 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter vgg16

### Case 4 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --cases-filter case_4 \
    --backbones-filter inception_v3

### Sonuçları Özetle (8 hücre tamamlandıktan sonra)

In [ ]:
# Tüm cell'leri tarayıp tek bir summary.md/csv üreten yardımcı betik (sonraki milestone'a kadar elle bu hücre yeter)
import json, pandas as pd
from pathlib import Path

rows = []
for case_dir in sorted(Path('outputs/03_baseline').glob('case_*')):
    for bb_dir in sorted(case_dir.glob('*')):
        m_path = bb_dir / 'metrics.json'
        if not m_path.exists():
            continue
        m = json.loads(m_path.read_text())
        row = {
            'case': m['case'], 'backbone': m['backbone'],
            'best_epoch': m['best_epoch'],
            'best_test_macro_f1': round(m['best_test_macro_f1'], 4),
            'best_val_macro_f1': round(m['best_val_macro_f1'], 4) if m['best_val_macro_f1'] == m['best_val_macro_f1'] else float('nan'),
        }
        for c, pc in m['best_test_metrics']['per_class'].items():
            row[f'F1__{c}'] = round(pc['f1'], 4)
        rows.append(row)
df = pd.DataFrame(rows)
Path('outputs/03_baseline').mkdir(parents=True, exist_ok=True)
df.to_csv('outputs/03_baseline/summary.csv', index=False)
Path('outputs/03_baseline/summary.md').write_text('# Baseline summary\n\n' + df.to_markdown(index=False))
df

## 4. Traditional augmentation - 4 Case x 2 Backbone

Plan D6 + paper sec 5.3 best transforms (offline): Rotation 270 + Shearing 30 + per-case 2nd rotation (Rot 90 for Case 1, Rot 45 for Cases 2/3/4). NF satirlarina aug uygulanmaz. internal_val ve test 100 percent gercek, outputs/02_splits/ altindan okunmaya devam eder.

### 4.1 Build the augmented training set

In [ ]:
# Her case icin: abnormal satirlara 3 transform uygula, PNG'leri outputs/04_traditional/<case>/aug_pngs/ altina kaydet,
# train_traditional.csv = real_rows + aug_rows uret. Idempotent - yeniden calistirmak guvenli.
!python scripts/04_build_traditional_set.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/04_traditional/summary.md

### 4.2 Smoke test

In [ ]:
# Phase 04 pipeline'i calisiyor mu? case_4 + VGG16 + 1 epoch.
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --cases-filter case_4 \
    --backbones-filter vgg16 \
    --epochs 1 \
    --out-tag 04_smoke

### Case 1 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_1 \
    --backbones-filter vgg16

### Case 1 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_1 \
    --backbones-filter inception_v3

### Case 2 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_2 \
    --backbones-filter vgg16

### Case 2 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_2 \
    --backbones-filter inception_v3

### Case 3 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_3 \
    --backbones-filter vgg16

### Case 3 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_3 \
    --backbones-filter inception_v3

### Case 4 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_4 \
    --backbones-filter vgg16

### Case 4 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_traditional \
    --train-csv-name train_traditional.csv \
    --out-tag 04_traditional \
    --cases-filter case_4 \
    --backbones-filter inception_v3

### Sonuclari Ozetle (8 hucre tamamlandiktan sonra)

In [ ]:
import json, pandas as pd
from pathlib import Path

def collect(out_tag):
    rows = []
    root = Path(f'outputs/{out_tag}')
    for case_dir in sorted(root.glob('case_*')):
        for bb_dir in sorted(case_dir.glob('*')):
            m_path = bb_dir / 'metrics.json'
            if not m_path.exists():
                continue
            m = json.loads(m_path.read_text())
            row = {
                'phase': out_tag, 'case': m['case'], 'backbone': m['backbone'],
                'best_epoch': m['best_epoch'],
                'best_test_macro_f1': round(m['best_test_macro_f1'], 4),
            }
            for c, pc in m['best_test_metrics']['per_class'].items():
                row[f'F1__{c}'] = round(pc['f1'], 4)
            rows.append(row)
    return pd.DataFrame(rows)

df_trad = collect('04_traditional')
df_trad.to_csv('outputs/04_traditional/summary.csv', index=False)
Path('outputs/04_traditional/summary.md').write_text('# Traditional summary\n\n' + df_trad.to_markdown(index=False))

df_base = collect('03_baseline')
if len(df_base) and len(df_trad):
    cmp = df_base[['case','backbone','best_test_macro_f1']].rename(columns={'best_test_macro_f1':'baseline'}).merge(
        df_trad[['case','backbone','best_test_macro_f1']].rename(columns={'best_test_macro_f1':'traditional'}),
        on=['case','backbone'])
    cmp['delta'] = (cmp['traditional'] - cmp['baseline']).round(4)
    print(cmp.to_string(index=False))
df_trad

## 5. WGAN per minority class - PAPER-FAITHFUL RESET (Wasserstein + weight clipping)

**Bu bolum, paper-faithful WGAN reset milestone'unun ana hücreleri.** Onceki WGAN-GP denemesinden sonra (DSN F1 traditional'in altinda kaldi) makaleye %100 sadik kalmaya geri donduk.

**Paper §4.6.2 + Figure 6 caption:** *Wasserstein loss with weight clipping*, DCGAN base (§4.6.1: discriminator has *convolutional, batch normalization, and Leaky ReLU*).

**Onceki -40M loss patlamasinin sebebi:** Critic'te BatchNorm yoktu + agirlik init'i (std=0.02) clip degerinin (0.01) ustunde idi -> critic iter 0'da sature basliyor, 6 katmanli network aktivasyonlari ust-uste cogaltarak ~3 milyara ulasiyor. Yeni kod hem critic'e BN ekledi hem de weight init'i uniform(-0.01, +0.01) ile clip araligina aldi. Loss artik O(1)'de kalmali.

**Sabit ayarlar (configs/wgan.yaml):** latent=120, image=256x256, RMSProp lr=5e-5, n_critic=5, weight_clip=0.01, total_iterations=20000, snapshots her 1500'de iter 5000'den itibaren.

**Skipped Osteophytes:** abnormal-majority (3201 train), generative oversampling gereksiz.

### 5.1 Smoke test (Vertebral collapse, 600 iters) - **paper-faithful weight clip**

Beklenti: `d_loss` magnitude `O(1)` ila `O(10)` arasinda kalmali (eski denemede -40M idi). `d_real_mean` ve `d_fake_mean` birbirinden temiz biçimde ayrilmali. Sample grid blob silüetler olabilir bu asamada - sadece generatorun sabit/uniform noise uretmedigini dogruluyoruz.

In [ ]:
# Pipeline calisiyor mu? En kucuk havuzlu sinifta (VC ~139 satir) hizli smoke.
# Iter 600 - 1 snapshot olusturmaz (first_snapshot_at=5000). Sadece pipeline'i ve sample png'leri uretir.
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --classes-filter "Vertebral collapse" \
    --iterations 600 \
    --out-tag 05_smoke
!ls outputs/05_smoke/case_1/vertebral_collapse/samples/ | head -5

### 5.2 Per-class training (paper-faithful, weight_clip)

**Sure tahmini (A100):** Iter basina ~0.4-0.6 sn (n_critic=5 dahil). 20K iter / sinif = ~2-3 saat / sinif. 6 sinif = ~12-18 saat - Colab Pro+ ile birden fazla oturuma yayilabilir.

Her sinif kendi hucresinde - tek bir oturumda da, parcali da kosturulabilir.

#### Disc space narrowing (case_1) - PAPER-FAITHFUL RE-TRAIN

Onceki WGAN-GP DSN ciktilari (outputs/05_wgan/case_1/disc_space_narrowing/) **yeniden yazilacak**. GP checkpointleri Phase 07/08'e tasinmamali - sadece yeni weight-clip checkpoint'leri kullanilacak.

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Disc space narrowing"

#### Vertebral collapse (case_1)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Vertebral collapse"

#### Foraminal stenosis (case_2)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Foraminal stenosis"

#### Spondylolysthesis (case_2)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Spondylolysthesis"

#### Surgical implant (case_3)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Surgical implant"

#### Other lesions (case_4)

In [ ]:
!python scripts/05_train_wgan.py \
    --config configs/base.yaml \
    --wgan configs/wgan.yaml \
    --out-tag 05_wgan \
    --classes-filter "Other lesions"

### 5.3 Quick QA - sample grids

In [ ]:
# En son uretilen sample grid'leri yan yana goster.
from pathlib import Path
from IPython.display import Image, display, HTML
for case_dir in sorted(Path('outputs/05_wgan').glob('case_*')):
    for cls_dir in sorted(case_dir.glob('*')):
        samples = sorted((cls_dir / 'samples').glob('iter_*.png'))
        if not samples:
            continue
        print(f'--- {case_dir.name}/{cls_dir.name} (last sample: {samples[-1].name}) ---')
        display(Image(str(samples[-1])))


## 6. Phase 07 - Synthetic image generation (DSN-only proof of concept)

Engineering decision: instead of training all 6 minority classes upfront (~30h A100), generate from the existing iter_006000.pt (Disc space narrowing, WGAN-GP) and run a partial Phase 08 on Case 1. If Case 1 macro F1 with WGAN-DSN augmentation moves above traditional Case 1 by >=0.02, train the remaining 5 classes. Otherwise we save 30 hours and pivot.

Output: outputs/07_wgan_generated/disc_space_narrowing/{pngs/, manifest.csv, manifest.json}

### 6.1 Generate ~800 synthetic DSN images from iter_006000.pt

In [ ]:
# Sayisi paper-style: real DSN train rows (537) * 1.48 ~ 800 sentetik. Argumanla artirilabilir.
!python scripts/07_generate_wgan.py \
    --config configs/base.yaml \
    --checkpoint outputs/05_wgan/case_1/disc_space_narrowing/checkpoints/iter_006000.pt \
    --n-samples 800 \
    --batch-size 32 \
    --out-tag 07_wgan_generated
!ls outputs/07_wgan_generated/disc_space_narrowing/pngs | head -5
!head -3 outputs/07_wgan_generated/disc_space_narrowing/manifest.csv

### 6.2 Quick QA - random 16 generated samples

In [ ]:
# Bir kac uretilen ornek goster - eski sample grid'lerinden farkli, fresh seed.
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

pngs = sorted(Path('outputs/07_wgan_generated/disc_space_narrowing/pngs').glob('*.png'))
sample = random.sample(pngs, k=min(16, len(pngs)))
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, p in zip(axes.flat, sample):
    ax.imshow(Image.open(p), cmap='gray')
    ax.axis('off')
plt.suptitle('WGAN-GP DSN @ iter 6000 - 16 random samples')
plt.tight_layout()
plt.show()

### 6.3 Decision gate (manual)

Before running Phase 08:
1. Check the 16-sample grid above. Are they **non-collapsed** (each one different) and **roughly spine-shaped**?
2. If yes -> proceed to Phase 08 (Case 1 hybrid_dsn_only).
3. If samples are mode-collapsed (all identical), pick a different checkpoint (e.g. iter_007500 if you have it) or revisit WGAN training.

Phase 08 will be added to this notebook in the next milestone once we know we want to proceed.

## 7. Phase 08 - Hybrid (DSN-only proof of concept on Case 1)

Recipe (paper sec 5.5): train_hybrid = real + WGAN gens + 3 geometric transforms applied to ALL abnormal images (real abn + WGAN gens). NF satirlarina aug uygulanmaz.

Bu adim **kismi** hybrid - sadece WGAN-DSN gen'leri ile. Eger Case 1 macro F1 traditional'a gore >=0.02 hareket ederse kalan 5 sinifi egit ve full hybrid yap. Hareket yoksa traditional'i tavan kabul edip kalan 30 saat A100'u harcama.

### 7.1 Build train_hybrid.csv for case_1 (dsn_only variant)

In [ ]:
# Real + real-aug rows outputs/04_traditional/case_1/train_traditional.csv'den geliyor.
# WGAN-DSN gen'leri ve uzerine 3 transform uygulanmis kopyalari ekleniyor.
!python scripts/08_build_hybrid_set.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --case case_1 \
    --wgan-classes disc_space_narrowing \
    --variant-tag dsn_only
!cat outputs/08_hybrid_dsn_only/case_1/summary.md

### 7.2 Train Case 1 / VGG16 with hybrid CSV

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_dsn_only \
    --train-csv-name train_hybrid.csv \
    --cases-filter case_1 \
    --backbones-filter vgg16 \
    --out-tag 08_hybrid_dsn_only

### 7.3 Train Case 1 / InceptionV3 with hybrid CSV

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_dsn_only \
    --train-csv-name train_hybrid.csv \
    --cases-filter case_1 \
    --backbones-filter inception_v3 \
    --out-tag 08_hybrid_dsn_only

### 7.4 Decision gate - karar protokolu

In [ ]:
# Baseline / traditional / hybrid_dsn_only - Case 1 icin Macro F1 karsilastir.
import json
from pathlib import Path
import pandas as pd

def collect_case_1(out_tag, label):
    rows = []
    root = Path(f'outputs/{out_tag}/case_1')
    if not root.exists():
        return None
    for bb_dir in sorted(root.glob('*')):
        m_path = bb_dir / 'metrics.json'
        if not m_path.exists():
            continue
        m = json.loads(m_path.read_text())
        rows.append({
            'phase': label,
            'backbone': m['backbone'],
            'best_epoch': m['best_epoch'],
            'macro_f1': round(m['best_test_macro_f1'], 4),
            'F1_DSN': round(m['best_test_metrics']['per_class']['Disc space narrowing']['f1'], 4),
            'F1_VC':  round(m['best_test_metrics']['per_class']['Vertebral collapse']['f1'], 4),
            'F1_NF':  round(m['best_test_metrics']['per_class']['No finding']['f1'], 4),
        })
    return pd.DataFrame(rows)

frames = [collect_case_1(t, l) for t, l in [
    ('03_baseline',           'baseline'),
    ('04_traditional',        'traditional'),
    ('08_hybrid_dsn_only',    'hybrid_dsn_only'),
]]
frames = [f for f in frames if f is not None]
comp = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if len(comp):
    print(comp.pivot(index='backbone', columns='phase', values='macro_f1'))
    print()
    print('Per-class F1 (Disc space narrowing):')
    print(comp.pivot(index='backbone', columns='phase', values='F1_DSN'))
    print()
    print('--- KARAR PROTOKOLU ---')
    pivot = comp.pivot(index='backbone', columns='phase', values='macro_f1')
    if 'traditional' in pivot.columns and 'hybrid_dsn_only' in pivot.columns:
        delta = (pivot['hybrid_dsn_only'] - pivot['traditional']).round(4)
        print('hybrid - traditional macro F1 deltas:')
        print(delta)
        max_delta = delta.max()
        if max_delta >= 0.02:
            print(f'\n>>> En iyi delta = +{max_delta}: WGAN sinyali var. Kalan 5 sinifi egitmeye DEGER.')
        elif max_delta > -0.005:
            print(f'\n>>> En iyi delta = {max_delta}: NOR HAREKET. Daha uzun WGAN egitimi (15-20K iter) dene veya traditional tavanı kabul et.')
        else:
            print(f'\n>>> En iyi delta = {max_delta}: WGAN F1 dusuruyor. Ya WGAN kalitesi yetersiz ya da bu task icin generative aug uygun degil.')
comp

## 8. Phase 08 - diagnostic experiments (Path B)

Hybrid_dsn_only sonucu traditional'in altinda kaldi (-0.0156 IC, -0.0208 VGG). 30 saatlik tam sweep'e gitmeden 2 ucuz teshis deneyi - WGAN'in *neden* zarar verdigini izole ediyoruz.

**Deney B1 (low dose):** 800 yerine 400 WGAN gen + transformlari (= 1600 sentetik). Gercek anormal:sentetik orani 1:2.4'e duser (su an 1:4.7). Hipotez: dozaj problemi.

**Deney B2 (no aug on wgan):** 800 WGAN gen, transform yok (= 800 sentetik). Gercek anormal:sentetik 1:1.2. Hipotez: bulanik sentetik uzerine rot/shear gurultusu katliyor.

### 8.1 Deney B1: low-dose hybrid (400 gens + transforms)

In [ ]:
!python scripts/08_build_hybrid_set.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --case case_1 \
    --wgan-classes disc_space_narrowing \
    --variant-tag dsn_lowdose \
    --max-wgan-per-class 400
!cat outputs/08_hybrid_dsn_lowdose/case_1/summary.md

#### B1 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_dsn_lowdose \
    --train-csv-name train_hybrid.csv \
    --cases-filter case_1 \
    --backbones-filter vgg16 \
    --out-tag 08_hybrid_dsn_lowdose

#### B1 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_dsn_lowdose \
    --train-csv-name train_hybrid.csv \
    --cases-filter case_1 \
    --backbones-filter inception_v3 \
    --out-tag 08_hybrid_dsn_lowdose

### 8.2 Deney B2: no-aug-on-wgan hybrid (800 gens, transform yok)

In [ ]:
!python scripts/08_build_hybrid_set.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --case case_1 \
    --wgan-classes disc_space_narrowing \
    --variant-tag dsn_no_synth_aug \
    --no-transforms-on-wgan
!cat outputs/08_hybrid_dsn_no_synth_aug/case_1/summary.md

#### B2 / VGG16

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_dsn_no_synth_aug \
    --train-csv-name train_hybrid.csv \
    --cases-filter case_1 \
    --backbones-filter vgg16 \
    --out-tag 08_hybrid_dsn_no_synth_aug

#### B2 / InceptionV3

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml \
    --cases configs/cases.yaml \
    --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_dsn_no_synth_aug \
    --train-csv-name train_hybrid.csv \
    --cases-filter case_1 \
    --backbones-filter inception_v3 \
    --out-tag 08_hybrid_dsn_no_synth_aug

### 8.3 Karar tablosu - 5 koşul yan yana

In [ ]:
import json
from pathlib import Path
import pandas as pd

def collect_case_1(out_tag, label):
    rows = []
    root = Path(f'outputs/{out_tag}/case_1')
    if not root.exists():
        return None
    for bb_dir in sorted(root.glob('*')):
        m_path = bb_dir / 'metrics.json'
        if not m_path.exists():
            continue
        m = json.loads(m_path.read_text())
        rows.append({
            'phase': label,
            'backbone': m['backbone'],
            'macro_f1': round(m['best_test_macro_f1'], 4),
            'F1_DSN': round(m['best_test_metrics']['per_class']['Disc space narrowing']['f1'], 4),
            'F1_VC':  round(m['best_test_metrics']['per_class']['Vertebral collapse']['f1'], 4),
        })
    return pd.DataFrame(rows)

configs = [
    ('03_baseline',                    'baseline'),
    ('04_traditional',                 'traditional'),
    ('08_hybrid_dsn_only',             'hybrid_full_dose'),
    ('08_hybrid_dsn_lowdose',          'hybrid_low_dose'),
    ('08_hybrid_dsn_no_synth_aug',     'hybrid_no_synth_aug'),
]
frames = [df for tag, lbl in configs if (df := collect_case_1(tag, lbl)) is not None]
comp = pd.concat(frames, ignore_index=True)
macro_pivot = comp.pivot(index='backbone', columns='phase', values='macro_f1')
ordered = [c for _, c in configs if c in macro_pivot.columns]
macro_pivot = macro_pivot[ordered]
print('Case 1 macro F1:')
print(macro_pivot)
print()
print('Case 1 DSN F1:')
print(comp.pivot(index='backbone', columns='phase', values='F1_DSN')[ordered])
print()
print('Case 1 VC F1:')
print(comp.pivot(index='backbone', columns='phase', values='F1_VC')[ordered])
print()
if 'traditional' in macro_pivot.columns:
    trad = macro_pivot['traditional']
    print('Macro F1 deltas vs traditional:')
    for col in ordered:
        if col == 'traditional':
            continue
        delta = (macro_pivot[col] - trad).round(4)
        print(f'  {col:25s}: {dict(delta)}')
    print()
    best_hybrid_col = max(
        (c for c in ordered if c.startswith('hybrid')),
        key=lambda c: (macro_pivot[c] - trad).max(),
        default=None,
    )
    if best_hybrid_col:
        best_delta = (macro_pivot[best_hybrid_col] - trad).max()
        print(f'En iyi hybrid varyant: {best_hybrid_col}, delta = {best_delta:+.4f}')
        if best_delta >= 0.02:
            print('>>> Recete bulundu. Bu varyantla kalan 5 sinifi egitmeye DEGER.')
        elif best_delta > -0.005:
            print('>>> Notr hareket. WGAN bu setupta isimize yaramiyor; traditional tavanini kabul et.')
        else:
            print('>>> Hicbir varyant traditional kadar iyi degil. WGAN yolu tikali; Yol A (yaz, dusunume gec).')
comp